# Met Office MOGREPS-G (Global Ensemble) – AWS ASDI Demo

This notebook demonstrates how to use the `site_archive_aws.MOGREPSGlobal` accessor
to read and visualise Met Office MOGREPS-G ensemble data from AWS S3.

## Dataset
MOGREPS-G is the Met Office's 18-member global ensemble prediction system,
producing forecasts at ~20 km grid spacing every 6 hours.
Each member represents a slightly different realisation of the initial conditions
and model physics, providing a probabilistic picture of the atmosphere.

## Requirements
```
pip install pyearthtools-archive-aws
```


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import site_archive_aws
from site_archive_aws import MOGREPSGlobal

print(f"site_archive_aws version: {site_archive_aws.__version__}")

## 1. Configure the Accessor

In [ ]:
QUERY_TIME = "2023-06-01T00:00"

# Load all 18 members for 2-m temperature
# For a quick demo, limit to the first 4 members
accessor = MOGREPSGlobal("2t", members=list(range(4)), anon=True)
print(accessor)

## 2. Load Ensemble Data

In [ ]:
ds = accessor[QUERY_TIME]
print(ds)
print("\nRealization (member) dimension:", ds["realization"].values)

# K → °C
t2m = ds["air_temperature"].squeeze(dim="time", drop=True) - 273.15
print("Shape (realization, lat, lon):", t2m.shape)

## 3. Plot Individual Ensemble Members

In [ ]:
n_members = len(t2m["realization"])
fig, axes = plt.subplots(
    2, 2,
    figsize=(16, 9),
    subplot_kw={"projection": ccrs.PlateCarree()},
)
axes = axes.flatten()

levels = np.linspace(-40, 40, 33)

for i, ax in enumerate(axes[:n_members]):
    member = int(t2m["realization"].values[i])
    field = t2m.isel(realization=i)

    im = ax.contourf(
        field["longitude"],
        field["latitude"],
        field.values,
        levels=levels,
        cmap="RdBu_r",
        transform=ccrs.PlateCarree(),
        extend="both",
    )
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
    ax.set_title(f"Member {member:03d}", fontsize=10)
    plt.colorbar(im, ax=ax, label="°C", shrink=0.8)

fig.suptitle(
    f"MOGREPS-G – 2-m Temperature (°C) – {QUERY_TIME}",
    fontsize=14, y=1.01,
)
plt.tight_layout()
plt.savefig("mogreps_g_members.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to mogreps_g_members.png")

## 4. Ensemble Mean and Spread

In [ ]:
ens_mean = t2m.mean("realization")
ens_std  = t2m.std("realization")   # spread (standard deviation)

fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(18, 6),
    subplot_kw={"projection": ccrs.PlateCarree()},
)

# Ensemble mean
im1 = ax1.contourf(
    ens_mean["longitude"], ens_mean["latitude"], ens_mean.values,
    levels=np.linspace(-40, 40, 33), cmap="RdBu_r",
    transform=ccrs.PlateCarree(), extend="both",
)
ax1.add_feature(cfeature.COASTLINE, linewidth=0.5)
plt.colorbar(im1, ax=ax1, label="°C", shrink=0.8)
ax1.set_title("Ensemble Mean – 2-m Temperature", fontsize=12)

# Ensemble spread
im2 = ax2.contourf(
    ens_std["longitude"], ens_std["latitude"], ens_std.values,
    levels=np.linspace(0, 10, 21), cmap="YlOrRd",
    transform=ccrs.PlateCarree(), extend="max",
)
ax2.add_feature(cfeature.COASTLINE, linewidth=0.5)
plt.colorbar(im2, ax=ax2, label="°C", shrink=0.8)
ax2.set_title("Ensemble Spread (Std Dev) – 2-m Temperature", fontsize=12)

fig.suptitle(f"MOGREPS-G – {QUERY_TIME} (UTC)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("mogreps_g_mean_spread.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to mogreps_g_mean_spread.png")

## 5. Summary

In this notebook we:
1. Loaded MOGREPS-G 2-m temperature for 4 ensemble members.
2. Plotted each member individually.
3. Computed and visualised the ensemble mean and spread.

To use all 18 members, set `members=None` (or omit the argument) when creating the accessor.

See `scripts/demo_mogreps_global.py` for the command-line equivalent.